# Experiment 1: Single Best Models (Baseline)

## Behaviour 
Logistic Regression\
Recall = 0.85\
F1 = 0.80

## Academic 
Random Forest\
Recall = 1.00\
F1 = 0.889


In [9]:
# check behaviour and academic validation sets aligned 
import pandas as pd

df_behaviour = pd.read_csv("../datasets/X_beh_val.csv")
df_academic = pd.read_csv("../datasets/X_aca_val.csv")

# Experiment 2:  Weighted Ensemble 
Behaviour probability = Pb\
Academic probability = Pa\

Average: ensemble_prob = (
    Pb +
    Pa
) / 2

Weighted combination: \
0.6 Academic\
0.4 Behaviour\
\
0.7 Academic\
0.3 Behaviour\
\
0.8 Academic\
0.2 Behaviour

## Objective

Combine the probability outputs from the Behaviour Risk Model and Academic Risk Model into a single student risk probability.

Unlike voting, this approach combines the predicted probabilities, which is much more appropriate for dashboard because the dashboard ultimately needs to display a risk score (%).

In [13]:
# load both final models 
import joblib
from pathlib import Path

# robust file path handling
model_dir = Path.cwd().parent / "models"

behaviour_model = joblib.load(model_dir / "behaviour_model_v1.pkl")
academic_model = joblib.load(model_dir / "academic_model_v1.pkl")

In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [17]:
# load validation datasets
dataset_dir = Path.cwd().parent / "datasets"

X_beh_val = pd.read_csv(dataset_dir / "X_beh_val.csv")
X_aca_val = pd.read_csv(dataset_dir / "X_aca_val.csv")

y_beh_val = pd.read_csv(dataset_dir / "y_beh_val.csv").squeeze()
y_aca_val = pd.read_csv(dataset_dir / "y_aca_val.csv").squeeze()

meta_beh_val = pd.read_csv(dataset_dir / "meta_beh_val.csv")
meta_aca_val = pd.read_csv(dataset_dir / "meta_aca_val.csv")

In [19]:
# load the behaviour scaler
scaler = joblib.load(model_dir / "behaviour_scaler_v1.pkl")
X_beh_val_scaled = scaler.transform(X_beh_val)

In [22]:
# predict probabilities 
Pb = behaviour_model.predict_proba(X_beh_val_scaled)[:, 1]

Pa = academic_model.predict_proba(X_aca_val)[:, 1]

In [21]:
meta_beh_val[
    ["student_key", "course_key", "academic_period_key"]
].equals(
    meta_aca_val[
        ["student_key", "course_key", "academic_period_key"]
    ]
)

True

In [23]:
beh_probs = meta_beh_val.copy()
beh_probs["behaviour_prob"] = Pb

aca_probs = meta_aca_val.copy()
aca_probs["academic_prob"] = Pa
aca_probs["actual"] = y_aca_val.values

In [24]:
ensemble_df = beh_probs.merge(
    aca_probs,
    on=["student_key", "course_key", "academic_period_key", "snapshot_date"],
    how="inner"
)

ensemble_df.shape

(34, 7)

In [28]:
# evaluate ensemble model with different weights
weights = [
    (0.8, 0.2),  # more behaviour
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),  # equal
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8)   # more academic
]

results = []

for w_beh, w_aca in weights:
    ensemble_prob = (
        w_beh * ensemble_df["behaviour_prob"]
        + w_aca * ensemble_df["academic_prob"]
    )

    ensemble_pred = (ensemble_prob >= 0.5).astype(int)
    y_true = ensemble_df["actual"]

    results.append({
        "model": f"Weighted Ensemble B{w_beh}_A{w_aca}",
        "behaviour_weight": w_beh,
        "academic_weight": w_aca,
        "accuracy": accuracy_score(y_true, ensemble_pred),
        "precision": precision_score(y_true, ensemble_pred, zero_division=0),
        "recall": recall_score(y_true, ensemble_pred, zero_division=0),
        "f1": f1_score(y_true, ensemble_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, ensemble_prob)
    })

weighted_results_df = pd.DataFrame(results)
weighted_results_df


,model,behaviour_weight,academic_weight,accuracy,precision,recall,f1,roc_auc
0,Weighted Ensemble B0.8_A0.2,0.8,0.2,0.852941,0.8,1.0,0.888889,0.939286
1,Weighted Ensemble B0.7_A0.3,0.7,0.3,0.852941,0.8,1.0,0.888889,0.939286
2,Weighted Ensemble B0.6_A0.4,0.6,0.4,0.852941,0.8,1.0,0.888889,0.935714
3,Weighted Ensemble B0.5_A0.5,0.5,0.5,0.852941,0.8,1.0,0.888889,0.935714
4,Weighted Ensemble B0.4_A0.6,0.4,0.6,0.852941,0.8,1.0,0.888889,0.921429
5,Weighted Ensemble B0.3_A0.7,0.3,0.7,0.852941,0.8,1.0,0.888889,0.921429
6,Weighted Ensemble B0.2_A0.8,0.2,0.8,0.852941,0.8,1.0,0.888889,0.910714


In [27]:
from sklearn.metrics import classification_report

best_row = weighted_results_df.sort_values(
    by=["recall", "f1", "roc_auc"],
    ascending=False
).iloc[0]

best_w_beh = best_row["behaviour_weight"]
best_w_aca = best_row["academic_weight"]

best_prob = (
    best_w_beh * ensemble_df["behaviour_prob"]
    + best_w_aca * ensemble_df["academic_prob"]
)

best_pred = (best_prob >= 0.5).astype(int)

print(confusion_matrix(ensemble_df["actual"], best_pred))
print(classification_report(ensemble_df["actual"], best_pred, zero_division=0))

[[ 9  5]
 [ 0 20]]
              precision    recall  f1-score   support

           0       1.00      0.64      0.78        14
           1       0.80      1.00      0.89        20

    accuracy                           0.85        34
   macro avg       0.90      0.82      0.84        34
weighted avg       0.88      0.85      0.85        34



# Experiment 3: Stacking
Inputs: Behaviour probability, Academic probability \
Meta model: Logistic Regression

Behaviour LR + Academic RF = Meta LR